In [1]:
# Motif enrichment for GR/NR3C1 in 1h IFNg-inducible ATAC peaks (GRanges input)

if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

bioc_pkgs <- c(
  "GenomicRanges",
  "SummarizedExperiment",
  "BSgenome.Hsapiens.UCSC.hg38",
  "JASPAR2022",
  "TFBSTools",
  "motifmatchr"
)

cran_pkgs <- c("dplyr", "tibble")

for (p in bioc_pkgs) {
  if (!requireNamespace(p, quietly = TRUE))
    BiocManager::install(p, ask = FALSE, update = FALSE)
}
for (p in cran_pkgs) {
  if (!requireNamespace(p, quietly = TRUE))
    install.packages(p)
}

suppressPackageStartupMessages({
  library(GenomicRanges)
  library(BSgenome.Hsapiens.UCSC.hg38)
  library(JASPAR2022)
  library(TFBSTools)
  library(motifmatchr)
  library(dplyr)
  library(tibble)
})
sessionInfo()

R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] tibble_3.2.1                      dplyr_1.1.4                      
 [3] motifmatchr_1.20.0                TFBSTools_1.36.0                 
 [5] JASPAR2022_0.99.7                 BiocFileCache_2.6.1              
 [7] dbplyr_2.3.4                      BSgenome.Hsapiens.UCSC

In [2]:
#read in 1h Dex-inducible peaks and define foreground/background sets
IFN_DAR <- read.delim("~/manuscripts/BAF_stimulus_CellGenomics/inputs/DARs/GM12878_stimulus/DAR_IFN_1h.tsv", sep='\t')
foreground <- makeGRangesFromDataFrame(IFN_DAR[IFN_DAR$change=="up",])
background <- makeGRangesFromDataFrame(IFN_DAR)

In [3]:
# Ensure standard chromosomes and remove NAs
foreground_gr <- keepStandardChromosomes(foreground, pruning.mode = "coarse")
universe_gr   <- keepStandardChromosomes(background, pruning.mode = "coarse")


# Define background-only as universe minus foreground (by exact matches)
background_gr <- universe_gr[!(universe_gr %in% foreground_gr)]

#Get NR3C1 motif (from JASPAR)
opts <- list(species = 9606, all_versions = FALSE)
pfms <- TFBSTools::getMatrixSet(JASPAR2022::JASPAR2022, opts)

# Match by motif name field (often contains "NR3C1" or "NR3C1::RXRA")
hit <- vapply(pfms, function(m) grepl("NR3C1", name(m), ignore.case = TRUE), logical(1))
cands <- pfms[hit]


# Prefer exact match if present, otherwise take first
cand_names <- vapply(cands, name, character(1))
exact_idx <- which(tolower(cand_names) == tolower("NR3C1"))
pfm <- if (length(exact_idx) > 0) cands[[exact_idx[1]]] else cands[[1]]
motif_id <- ID(pfm)
motif_name <- name(pfm)

pwms <- PFMatrixList(pfm)

In [4]:
# count motif presence in foreground and background
genome = BSgenome.Hsapiens.UCSC.hg38
p_cutoff=1e-4

get_match_matrix <- function(se) {
    a <- SummarizedExperiment::assays(se)
    nm <- names(a)
    if (length(nm) == 0) stop("No assays found in matchMotifs() result.")
    # Common names across versions
    preferred <- c("matches", "motifMatches", "motif_matches", "match", "binary")
    pick <- intersect(preferred, nm)
    assay_name <- if (length(pick) > 0) pick[1] else nm[1]
    message("Using assay: ", assay_name)
    SummarizedExperiment::assay(se, assay_name)
}
                
fg_matches <- motifmatchr::matchMotifs(
    pwms = pwms, subject = foreground_gr, genome = genome,
    p.cutoff = p_cutoff, out = "scores" 
)
bg_matches <- motifmatchr::matchMotifs(
    pwms = pwms, subject = background_gr, genome = genome,
    p.cutoff = p_cutoff, out = "scores" 
)

    fg_mat <- get_match_matrix(fg_matches)
    bg_mat <- get_match_matrix(bg_matches)
                
    fg_has <- as.logical(fg_mat[, 1])
    bg_has <- as.logical(bg_mat[, 1])

 

Using assay: motifMatches

Using assay: motifMatches



In [7]:
 # 2x2 table:
  #               motif+   motif-
  # foreground       a       b
  # background       c       d
  a <- sum(fg_has)
  b <- sum(!fg_has)
  c <- sum(bg_has)
  d <- sum(!bg_has)

  mat <- matrix(c(a, b, c, d), nrow = 2, byrow = TRUE,
                dimnames = list(Set = c("Foreground", "Background"),
                                Motif = c("Motif+", "Motif-")))

  ft <- fisher.test(mat, alternative="greater")

  # Helpful effect sizes
  fg_rate <- a / (a + b)
  bg_rate <- c / (c + d)
  risk_ratio <- (fg_rate / bg_rate)

tibble(
    motif_id = motif_id,
    motif_name = motif_name,
   # motif_pattern = motif_pattern,
    p = ft$p.value,
    n_foreground = a + b,
    n_background = c + d,
    fg_motif_pos = a,
    bg_motif_pos = c,
    fg_hit_rate = fg_rate,
    bg_hit_rate = bg_rate,
    risk_ratio = risk_ratio,
    odds_ratio = unname(ft$estimate),
    fisher_p = ft$p.value,
    conf_low = ft$conf.int[1],
    conf_high = ft$conf.int[2]
  ) %>%
    mutate(across(where(is.numeric), ~ ifelse(is.nan(.x) | is.infinite(.x), NA_real_, .x))) %>%
    write.table("NR3C1_enr_IFN-up-1h.tsv", sep='\t', row.names=F)